# ReciclaIA – Verificación final antes de presentar
Hace **todo lo que faltaba comprobar** y deja listo el material para la presentación:
1. **Fuga de datos:** ¿alguna foto de train es casi igual a una del val, del test o del examen? (pHash)
2. **Si hay fuga en v9** (más del 1%): quita esas fotos y **reentrena v9 limpio** (~25 min, a prueba de cortes).
3. **Métricas limpias:** val y test **sin** fotos parecidas a train; robustez (grises, fondo, fondos vacíos, falsos Rojo);
   fotos reales de celular; examen del docente.
4. Deja el **modelo final** como `modelos/clasificador.pt` y guarda tablas, gráficos y un resumen en `revision_final/`.

**GPU T4 → Entorno de ejecución → Ejecutar todo.** Sin fuga: ~15 min. Con fuga: ~40 min.
Si se corta: **Ejecutar todo** otra vez (lo pesado queda guardado en Drive).

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" imagehash tabulate
from google.colab import drive
drive.mount("/content/drive")
import os, re, json, shutil, zipfile, hashlib
from datetime import datetime
import numpy as np, pandas as pd, torch, imagehash
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from sklearn.metrics import f1_score, confusion_matrix

DRIVE_DIR = "/content/drive/MyDrive/data basurita"
MODELOS, RUNS = f"{DRIVE_DIR}/modelos", f"{DRIVE_DIR}/runs_clasificador"
FINAL = f"{DRIVE_DIR}/revision_final"
CACHE_DS = f"{RUNS}/v9r_dataset.zip"
V9 = f"{RUNS}/v9r/weights/best.pt"
MEZCLA30 = f"{DRIVE_DIR}/revision_v8_mezcla/mezcla30.pt"
MODELO_V5 = f"{MODELOS}/clasificador_v5.pt"
AJUSTE_V8B = {"Amarillo": 0.25, "Azul": 0.50, "Gris": 0.50, "Marron": 0.00, "Negro": -1.00, "Rojo": 1.75}
DS, SEMILLA, UMBRAL_FUGA = "/content/ds", 42, 0.01
EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
NOMBRES = {"Azul": "azul_papel_carton", "Amarillo": "amarillo_metales", "Blanco": "blanco_plasticos",
           "Gris": "gris_vidrio", "Marron": "marron_organicos", "Rojo": "rojo_peligrosos", "Negro": "negro_generales"}
LARGO_A_CORTO = {v: k for k, v in NOMBRES.items()}
COLORES = list(NOMBRES)
imagenes = lambda d: sorted(f for f in os.listdir(d) if f.lower().endswith(EXT)) if os.path.isdir(d) else []
for r in (CACHE_DS, V9, MEZCLA30, MODELO_V5, f"{DRIVE_DIR}/examen_test.zip"):
    assert os.path.exists(r), f"Falta en Drive: {r}"
os.makedirs(FINAL, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU (Entorno -> T4)")

## 1. Datos: dataset de v9 (desde Drive), examen y fotos de celular

In [ ]:
shutil.rmtree(DS, ignore_errors=True)
zipfile.ZipFile(CACHE_DS).extractall("/content")
GRUPOS = {
    "cardboard": ("garbage", "Azul (papel/cartón)", {"Azul"}, True), "paper": ("garbage", "Azul (papel/cartón)", {"Azul"}, True),
    "metal": ("garbage", "Amarillo (metal)", {"Amarillo"}, True), "plastic": ("garbage", "Blanco (plástico)", {"Blanco"}, True),
    "glass": ("garbage", "Gris (vidrio)", {"Gris"}, True), "organik": ("garbage_v4", "Marrón (Organik)", {"Marron"}, True),
    "anorganik": ("garbage_v4", "Anorganik", {"Amarillo", "Azul", "Blanco", "Gris"}, False),
    "residu": ("garbage_v4", "Residu (Negro/Rojo)", {"Negro", "Rojo"}, False),
    "infectious": ("medico", "Rojo (biocontaminado)", {"Rojo"}, True),
    "general": ("medico", "No-Rojo (médico general)", {"Amarillo", "Azul", "Blanco", "Gris", "Marron", "Negro"}, False)}
shutil.rmtree("/content/examen", ignore_errors=True)
zipfile.ZipFile(f"{DRIVE_DIR}/examen_test.zip").extractall("/content/examen")
filas = []
for raiz, _, archivos in os.walk("/content/examen"):
    c = os.path.basename(raiz).lower()
    if "test" in raiz.replace("\\", "/").split("/") and c in GRUPOS:
        filas += [(os.path.join(raiz, f), *GRUPOS[c], c) for f in sorted(archivos) if f.lower().endswith(EXT)]
EXAMEN = pd.DataFrame(filas, columns=["ruta", "dataset", "grupo", "colores_ok", "estricta", "clase"])
CAMARA = "/content/test_camara"
shutil.rmtree(CAMARA, ignore_errors=True)
for corto, largo in NOMBRES.items():
    os.makedirs(f"{CAMARA}/{largo}")
    for f in imagenes(f"{DRIVE_DIR}/test_camara/{corto}"):
        shutil.copy2(f"{DRIVE_DIR}/test_camara/{corto}/{f}", f"{CAMARA}/{largo}/{f}")

def lista(split, filtro=None):
    return [f"{DS}/{split}/{c}/{f}" for c in NOMBRES.values() for f in imagenes(f"{DS}/{split}/{c}") if filtro is None or filtro(f)]
TRAIN_NUEVAS = lista("train", lambda f: f.startswith("v9r_"))
TRAIN_ORIG = lista("train", lambda f: not f.startswith("v9r_") and not f.startswith("fondo_"))
VAL_ALL, TEST_ALL = lista("val"), lista("test")
print(f"Train: {len(TRAIN_ORIG)} originales + {len(TRAIN_NUEVAS)} nuevas de v9 | Val {len(VAL_ALL)} | Test {len(TEST_ALL)} | "
      f"Examen {len(EXAMEN)} (deben ser 1455) | Celular {sum(len(imagenes(f'{CAMARA}/{c}')) for c in NOMBRES.values())}")

## 2. Huellas pHash (se guardan en Drive para no recalcular)

In [ ]:
HUELLAS_F = f"{FINAL}/huellas.json"
memo = json.load(open(HUELLAS_F)) if os.path.exists(HUELLAS_F) else {}
def clave(r):
    return r.replace("/content/", "")
def huellas(rutas):
    nuevas = 0
    for r in rutas:
        k = clave(r)
        if k not in memo:
            try:
                memo[k] = str(imagehash.phash(Image.open(r).convert("RGB")))
            except Exception:
                memo[k] = None
            nuevas += 1
    if nuevas:
        json.dump(memo, open(HUELLAS_F, "w"))
    return np.array([int(memo[clave(r)], 16) if memo[clave(r)] else 0 for r in rutas], dtype=np.uint64)
H = {n: huellas(x) for n, x in (("train_nuevas", TRAIN_NUEVAS), ("train_orig", TRAIN_ORIG), ("val", VAL_ALL),
                                 ("test", TEST_ALL), ("examen", list(EXAMEN.ruta)))}

def dist_min(A, B, bloque=256):
    # distancia de Hamming minima de cada huella de A contra todas las de B
    out = np.full(len(A), 64)
    if len(B) == 0:
        return out
    for i in range(0, len(A), bloque):
        x = np.bitwise_xor(A[i:i + bloque, None], B[None, :])
        out[i:i + bloque] = np.unpackbits(x.view(np.uint8), axis=-1).reshape(x.shape[0], x.shape[1], 64).sum(-1).min(1)
    return out
print("Huellas listas")

## 3. Comprobación de fuga de datos

In [ ]:
INFORME_FUGA, D = [], {}
for objetivo in ("val", "test"):
    for fuente in ("train_nuevas", "train_orig"):
        d = dist_min(H[objetivo], H[fuente]); D[(objetivo, fuente)] = d
        INFORME_FUGA.append({"comparacion": f"{objetivo} vs {fuente}", "fotos": len(d),
                             "casi_iguales_<=4": int((d <= 4).sum()), "pct_<=4": round(float((d <= 4).mean()) * 100, 2),
                             "parecidas_<=8": int((d <= 8).sum())})
for fuente in ("train_nuevas", "train_orig"):
    d = dist_min(H["examen"], H[fuente]); D[("examen", fuente)] = d
    INFORME_FUGA.append({"comparacion": f"examen vs {fuente}", "fotos": len(d),
                         "casi_iguales_<=4": int((d <= 4).sum()), "pct_<=4": round(float((d <= 4).mean()) * 100, 2),
                         "parecidas_<=8": int((d <= 8).sum())})
FUGA = pd.DataFrame(INFORME_FUGA)
print(FUGA.to_string(index=False))
FUGA.to_csv(f"{FINAL}/fuga_de_datos.csv", index=False)
d_ex = D[("examen", "train_orig")]
if (d_ex <= 4).any():
    print("\nExamen casi igual a la particion ORIGINAL por grupo (afecta a TODAS las versiones, viene del dataset v1):")
    print(EXAMEN.assign(casi=d_ex <= 4).groupby("grupo").casi.sum().to_string())

FUGA_V9 = max(float((D[("val", "train_nuevas")] <= 4).mean()), float((D[("test", "train_nuevas")] <= 4).mean()))
print(f"\nFuga de v9 (fotos nuevas de train casi iguales a val/test): {FUGA_V9:.2%}  ->  "
      + ("OK, menor al 1%" if FUGA_V9 < UMBRAL_FUGA else "MAYOR al 1%: se reentrena v9 sin esas fotos"))

## 4. Solo si hubo fuga: reentrenar v9 limpio (desde la mezcla de v8b, a prueba de cortes)

In [ ]:
V9_LIMPIO = None
if FUGA_V9 >= UMBRAL_FUGA:
    # fotos nuevas de train casi iguales a alguna de val/test
    d_inv = np.minimum(dist_min(H["train_nuevas"], H["val"]), dist_min(H["train_nuevas"], H["test"]))
    malas = {TRAIN_NUEVAS[i] for i in np.nonzero(d_inv <= 8)[0]}
    for r in malas:
        for p in (r, os.path.join(os.path.dirname(r), "fondo_" + os.path.basename(r))):
            if os.path.exists(p):
                os.remove(p)
    print(f"Quitadas de train: {len(malas)} fotos nuevas (y sus copias sobre fondo)")
    V9_LIMPIO = f"{RUNS}/v9r_limpio/weights/best.pt"
    ULT = f"{RUNS}/v9r_limpio/weights/last.pt"
    if os.path.exists(ULT) and torch.load(ULT, map_location="cpu", weights_only=False).get("epoch") == -1:
        print("v9 limpio ya estaba entrenado -> se reutiliza")
    elif os.path.exists(ULT):
        YOLO(ULT).train(resume=True)
    else:
        YOLO(MEZCLA30).train(data=DS, epochs=10, imgsz=224, batch=64, optimizer="SGD", lr0=0.0005, lrf=0.2,
                             warmup_epochs=0, patience=4, seed=SEMILLA, project=RUNS, name="v9r_limpio", exist_ok=True)
else:
    print("No hace falta reentrenar")

## 5. Evaluación completa con métricas limpias

In [ ]:
def hornear_dict(ruta, ajuste, salida):
    ck = torch.load(ruta, map_location="cpu", weights_only=False)
    m = (ck["ema"] if ck.get("ema") is not None else ck["model"]).float()
    for i, n in m.names.items():
        m.model[-1].linear.bias.data[i] += ajuste.get(LARGO_A_CORTO[n], 0.0)
    ck["model"], ck["ema"] = m.half(), None
    torch.save(ck, salida)
    return salida

def predecir(ruta_modelo, rutas):
    m = YOLO(ruta_modelo); rutas = list(rutas); P = []
    for i in range(0, len(rutas), 128):
        P += [r.probs.data.cpu().numpy() for r in m.predict(rutas[i:i + 128], imgsz=224, verbose=False, stream=True)]
    orden = np.array([LARGO_A_CORTO[m.names[i]] for i in range(len(m.names))])
    return orden[np.array(P).argmax(1)] if P else np.array([])

def reales(rutas):
    return np.array([LARGO_A_CORTO[r.split("/")[-2]] for r in rutas])

def balanceado(y, p):
    return float(np.mean([(p[y == c] == c).mean() for c in np.unique(y)]))

# val y test LIMPIOS: sin fotos casi iguales (<=4) a cualquier foto de train
d_val = np.minimum(D[("val", "train_nuevas")], D[("val", "train_orig")])
d_test = np.minimum(D[("test", "train_nuevas")], D[("test", "train_orig")])
VAL = [r for r, d in zip(VAL_ALL, d_val) if d > 4]
TEST = [r for r, d in zip(TEST_ALL, d_test) if d > 4]
print(f"Val limpio: {len(VAL)} de {len(VAL_ALL)} | Test limpio: {len(TEST)} de {len(TEST_ALL)}")

# pruebas de robustez sobre el val limpio
rng = np.random.default_rng(SEMILLA + 1)
def fondo_aleatorio(lado):
    color = rng.integers(0, 256, 3).astype(np.float32); tipo = rng.integers(4)
    yy, xx = np.mgrid[0:lado, 0:lado].astype(np.float32) / lado
    if tipo == 0:
        a = np.ones((lado, lado, 1)) * color + rng.normal(0, 6, (lado, lado, 3))
    elif tipo == 1:
        a = color * (0.6 + 0.6 * (xx if rng.random() < .5 else yy)[..., None])
    elif tipo == 2:
        a = color * (0.85 + 0.15 * np.sin(yy * rng.uniform(20, 60) + np.sin(xx * 6) * 2)[..., None]) + rng.normal(0, 4, (lado, lado, 3))
    else:
        a = np.full((lado, lado, 3), rng.uniform(40, 220)) + rng.normal(0, 5, (lado, lado, 3))
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
def sobre_fondo(im, lado=320):
    lienzo = fondo_aleatorio(lado); esc = rng.uniform(0.35, 0.70)
    w = int(lado * esc); h = int(w * im.height / max(1, im.width))
    if h > lado * 0.9:
        h = int(lado * 0.9); w = int(h * im.width / max(1, im.height))
    pieza = im.resize((max(16, w), max(16, h)))
    lienzo.paste(pieza, (int(rng.integers(0, lado - pieza.width + 1)), int(rng.integers(0, lado - pieza.height + 1))))
    return lienzo
for d in ("/content/vg", "/content/vf", "/content/vacios", "/content/exg"):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
VAL_G, VAL_F = [], []
for i, r in enumerate(VAL):
    im = Image.open(r).convert("RGB"); c = r.split("/")[-2]
    os.makedirs(f"/content/vg/{c}", exist_ok=True); os.makedirs(f"/content/vf/{c}", exist_ok=True)
    g, f = f"/content/vg/{c}/{i}.jpg", f"/content/vf/{c}/{i}.jpg"
    im.convert("L").convert("RGB").save(g); sobre_fondo(im).save(f); VAL_G.append(g); VAL_F.append(f)
VACIOS = []
for i in range(300):
    p = f"/content/vacios/{i}.jpg"; fondo_aleatorio(320).save(p); VACIOS.append(p)
V1I = EXAMEN[EXAMEN.dataset == "garbage"]; EXG = []
for i, r in enumerate(V1I.ruta):
    p = f"/content/exg/{i}.jpg"; Image.open(r).convert("L").convert("RGB").save(p); EXG.append(p)
CAM = [f"{CAMARA}/{c}/{f}" for c in NOMBRES.values() for f in imagenes(f"{CAMARA}/{c}")]

def hora(n):
    m = re.search(r"(\d{8}_\d{6})", n)
    return datetime.strptime(m.group(1), "%Y%m%d_%H%M%S") if m else None
def objetos(rutas, preds):
    df, ok = pd.DataFrame({"n": [os.path.basename(r) for r in rutas], "y": reales(rutas), "p": preds}), []
    for c, g in df.groupby("y"):
        fotos = sorted(g.n, key=lambda f: (hora(f) or datetime.min, f)); grupos, prev = [], None
        for f in fotos:
            t = hora(f)
            if prev is None or t is None or (t - prev).total_seconds() > 60:
                grupos.append([])
            grupos[-1].append(f); prev = t
        ok += [g[g.n.isin(gr)].p.mode()[0] == c for gr in grupos]
    return sum(ok), len(ok)

def evaluar(ruta):
    yv, yt = reales(VAL), reales(TEST)
    pv, pg, pf, pt = predecir(ruta, VAL), predecir(ruta, VAL_G), predecir(ruta, VAL_F), predecir(ruta, TEST)
    pvac, pc = predecir(ruta, VACIOS), predecir(ruta, CAM)
    ex = EXAMEN.assign(pred=predecir(ruta, EXAMEN.ruta)); ex["ok"] = [p in o for p, o in zip(ex.pred, ex.colores_ok)]
    grupos = ex.groupby("grupo").ok.mean()
    est = ex[ex.estricta]; y = est.colores_ok.map(lambda s: next(iter(s)))
    mf1 = f1_score(y, est.pred, labels=sorted(y.unique()), average="macro", zero_division=0)
    med = ex[ex.dataset == "medico"]
    rec = (med[med.clase == "infectious"].pred == "Rojo").mean()
    rojos = med[med.pred == "Rojo"]; prec = (rojos.clase == "infectious").mean() if len(rojos) else 0
    f1r = 2 * prec * rec / (prec + rec) if prec + rec else 0
    org = ex[ex.clase == "organik"].ok.mean()
    caida = (ex[ex.dataset == "garbage"].ok.mean() - np.mean([p in o for p, o in zip(predecir(ruta, EXG), V1I.colores_ok)])) * 100
    base = 8 * grupos.mean() + 4 * mf1 + 3 + (3 - 0.05 * caida); seg = 3 * org + 28 / 9 * f1r
    oc, on = objetos(CAM, pc)
    fila = {"val_limpio_color": balanceado(yv, pv), "val_limpio_grises": balanceado(yv, pg), "val_limpio_con_fondo": balanceado(yv, pf),
            "test_limpio": float((pt == yt).mean()), "fondos_vacios_dice_Rojo": float((pvac == "Rojo").mean()),
            "falsos_Rojo_con_fondo": float((pf[yv != "Rojo"] == "Rojo").mean()),
            "celular_fotos": float((pc == reales(CAM)).mean()), "celular_objetos": f"{oc}/{on}",
            **{f"examen: {k}": round(v, 3) for k, v in grupos.items()},
            "nota_estimada": round(base + seg, 2), "nota_si_seguridad_max2": round(base + min(seg, 2), 2)}
    fila["puntaje_val"] = (fila["val_limpio_color"] + fila["val_limpio_grises"] + fila["val_limpio_con_fondo"]) / 3
    return fila, {"test_y": yt, "test_p": pt, "examen": ex}

V8B = hornear_dict(MEZCLA30, AJUSTE_V8B, "/content/v8b.pt")
CANDIDATOS = {"v5": MODELO_V5, "v8b": V8B, "v9": V9}
if V9_LIMPIO and os.path.exists(V9_LIMPIO):
    CANDIDATOS["v9_limpio"] = V9_LIMPIO
RES, DET = {}, {}
for n, r in CANDIDATOS.items():
    RES[n], DET[n] = evaluar(r)
    print("evaluado", n)
TABLA = pd.DataFrame(RES)
pd.set_option("display.width", 220)
print(TABLA.to_string())
TABLA.to_csv(f"{FINAL}/comparacion_final.csv")

## 6. Modelo final y material para la presentación

In [ ]:
opciones = [k for k in CANDIDATOS if k.startswith("v9")]
FINAL_N = max(opciones, key=lambda k: RES[k]["puntaje_val"])
shutil.copy(CANDIDATOS[FINAL_N], f"{MODELOS}/clasificador.pt")
shutil.copy(CANDIDATOS[FINAL_N], f"{MODELOS}/clasificador_final.pt")
md5 = hashlib.md5(open(f"{MODELOS}/clasificador.pt", "rb").read()).hexdigest()
print(f"Modelo final: {FINAL_N} -> modelos/clasificador.pt y clasificador_final.pt | md5 {md5}")

# Graficos
det = DET[FINAL_N]
cm = confusion_matrix(det["test_y"], det["test_p"], labels=COLORES, normalize="true")
fig, ax = plt.subplots(figsize=(7, 6)); im = ax.imshow(cm, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(7)); ax.set_xticklabels(COLORES, rotation=45); ax.set_yticks(range(7)); ax.set_yticklabels(COLORES)
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cm[i, j] > .5 else "black")
ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.set_title(f"Matriz de confusión – test limpio ({FINAL_N})")
plt.colorbar(im); plt.tight_layout(); plt.savefig(f"{FINAL}/matriz_confusion_test.png", dpi=160); plt.show()

grupos_ex = [k for k in TABLA.index if k.startswith("examen: ")]
fig, ax = plt.subplots(figsize=(10, 4.5)); x = np.arange(len(grupos_ex)); w = 0.8 / len(CANDIDATOS)
for i, n in enumerate(CANDIDATOS):
    ax.bar(x + i * w, [RES[n][g] for g in grupos_ex], w, label=n)
ax.set_xticks(x + w * (len(CANDIDATOS) - 1) / 2); ax.set_xticklabels([g.replace("examen: ", "") for g in grupos_ex], rotation=30, ha="right")
ax.set_ylim(0, 1); ax.set_ylabel("Acierto"); ax.set_title("Examen del docente por grupo"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FINAL}/examen_por_grupo.png", dpi=160); plt.show()

rob = ["val_limpio_color", "val_limpio_grises", "val_limpio_con_fondo", "test_limpio", "celular_fotos"]
fig, ax = plt.subplots(figsize=(9, 4)); x = np.arange(len(rob))
for i, n in enumerate(CANDIDATOS):
    ax.bar(x + i * w, [RES[n][k] for k in rob], w, label=n)
ax.set_xticks(x + w * (len(CANDIDATOS) - 1) / 2); ax.set_xticklabels(["Val color", "Val grises", "Val con fondo", "Test", "Fotos celular"])
ax.set_ylim(0, 1); ax.set_title("Robustez (datos limpios)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FINAL}/robustez.png", dpi=160); plt.show()

# Resumen para el informe / la presentacion
r = RES[FINAL_N]
lineas = [f"# ReciclaIA – Resultados finales ({datetime.now():%Y-%m-%d %H:%M})", "",
          f"**Modelo final:** {FINAL_N} (YOLO11n-cls) · md5 `{md5}`", "",
          "## Fuga de datos", FUGA.to_markdown(index=False), "",
          f"Fuga de v9 (fotos nuevas de train casi iguales a val/test): **{FUGA_V9:.2%}** "
          + ("(menor al 1%: aceptable)" if FUGA_V9 < UMBRAL_FUGA else "(mayor al 1%: se reentrenó v9 sin esas fotos)"), "",
          f"Val limpio: {len(VAL)} de {len(VAL_ALL)} fotos · Test limpio: {len(TEST)} de {len(TEST_ALL)} fotos "
          "(se quitaron las que se parecen a alguna foto de train).", "",
          "## Resultados del modelo final",
          f"- Val limpio a color: {r['val_limpio_color']:.1%} · en grises: {r['val_limpio_grises']:.1%} · con fondo: {r['val_limpio_con_fondo']:.1%}",
          f"- **Test limpio: {r['test_limpio']:.1%}**",
          f"- Fotos reales de celular: {r['celular_fotos']:.1%} · objetos {r['celular_objetos']}",
          f"- Falsos Rojo (objeto no peligroso sobre fondo liso): {r['falsos_Rojo_con_fondo']:.1%}",
          f"- Fondos vacíos llamados Rojo: {r['fondos_vacios_dice_Rojo']:.1%}",
          f"- Nota estimada (fórmula del docente): {r['nota_estimada']} (optimista: fuentes del mismo origen que el examen)", "",
          "## Comparación", TABLA.round(3).to_markdown(), "",
          "## Declaración",
          "- Fuentes del mismo origen que el examen (TrashNet, Waste Classification, Medical Waste 4.0) usadas SIN sus fotos de test "
          "(filtro por nombre, sesión ±20 s y pHash ≤12).",
          "- Algunas decisiones intermedias (v7–v8b) se tomaron mirando el examen; v9 se eligió con validación y reglas fijas.",
          "- La medida más honesta es el **test limpio** y las **fotos reales de celular**."]
open(f"{FINAL}/resumen_final.md", "w").write("\n".join(lineas))
print("\n".join(lineas[:22]))
print(f"\nTodo guardado en {FINAL}: resumen_final.md, comparacion_final.csv, fuga_de_datos.csv, "
      "matriz_confusion_test.png, examen_por_grupo.png, robustez.png")